In [6]:
import numpy as np
from scipy.special import erf, j0, gammaln, expi, gamma, beta, gammainc, betainc, stdtr, stdtrit

In [2]:
def integMonteCarlo(func, dim, limInf, limSup, numMuestras, semilla=None):
    genAleat = np.random.default_rng(semilla)
    vol = np.prod(limSup - limInf)
    sumaF = 0.0
    sumaF2 = 0.0

    for _ in range(numMuestras):
        punto = limInf + genAleat.random(dim) * (limSup - limInf)
        valor = func(punto)
        sumaF += valor
        sumaF2 += valor * valor

    promF = sumaF / numMuestras
    promF2 = sumaF2 / numMuestras
    varianza = (promF2 - promF**2) / numMuestras
    errEst = vol * np.sqrt(max(varianza, 0.0))
    intEst = vol * promF

    return intEst, errEst

In [3]:
# Caso 1: Integral de erf(x) en [0, 1] (1D)
# erf no tiene antiderivada elemental
def func1(x):
    return erf(x[0])

res1, err1 = integMonteCarlo(func1, dim=1, limInf=np.array([0.0]), 
                              limSup=np.array([1.0]), numMuestras=100000, semilla=42)
print(f"Caso 1 - erf(x) en [0,1]: {res1:.6f} ± {err1:.6f}")


# Caso 2: Integral de J₀(x) en [0, 5] (1D)
# Función de Bessel de primera especie, sin primitiva elemental
def func2(x):
    return j0(x[0])

res2, err2 = integMonteCarlo(func2, dim=1, limInf=np.array([0.0]), 
                              limSup=np.array([5.0]), numMuestras=100000, semilla=42)
print(f"Caso 2 - J₀(x) en [0,5]: {res2:.6f} ± {err2:.6f}")


# Caso 3: Integral de ln(Γ(x)) en [1, 3] (1D)
# Log-gamma, función especial sin antiderivada elemental
def func3(x):
    return gammaln(x[0])

res3, err3 = integMonteCarlo(func3, dim=1, limInf=np.array([1.0]), 
                              limSup=np.array([3.0]), numMuestras=100000, semilla=42)
print(f"Caso 3 - ln(Γ(x)) en [1,3]: {res3:.6f} ± {err3:.6f}")


# Caso 4: Integral doble de exp(-x²-y²)·cos(xy) en [0,1]×[0,1] (2D)
# Producto de gaussiana y oscilación, sin primitiva cerrada
def func4(x):
    return np.exp(-(x[0]**2 + x[1]**2)) * np.cos(x[0] * x[1])

res4, err4 = integMonteCarlo(func4, dim=2, limInf=np.array([0.0, 0.0]), 
                              limSup=np.array([1.0, 1.0]), numMuestras=200000, semilla=42)
print(f"Caso 4 - exp(-x²-y²)·cos(xy) en [0,1]²: {res4:.6f} ± {err4:.6f}")


# Caso 5: Integral triple de sin(x·y·z) / (1 + x² + y² + z²) en [0,2]³ (3D)
# Función oscilatoria racional, sin antiderivada elemental en 3D
def func5(x):
    xyz = x[0] * x[1] * x[2]
    denom = 1.0 + x[0]**2 + x[1]**2 + x[2]**2
    return np.sin(xyz) / denom if denom > 1e-10 else 0.0

res5, err5 = integMonteCarlo(func5, dim=3, limInf=np.array([0.0, 0.0, 0.0]), 
                              limSup=np.array([2.0, 2.0, 2.0]), numMuestras=300000, semilla=42)
print(f"Caso 5 - sin(xyz)/(1+x²+y²+z²) en [0,2]³: {res5:.6f} ± {err5:.6f}")

Caso 1 - erf(x) en [0,1]: 0.486647 ± 0.000788
Caso 2 - J₀(x) en [0,5]: 0.707504 ± 0.008013
Caso 3 - ln(Γ(x)) en [1,3]: 0.224950 ± 0.001527
Caso 4 - exp(-x²-y²)·cos(xy) en [0,1]²: 0.540373 ± 0.000516
Caso 5 - sin(xyz)/(1+x²+y²+z²) en [0,2]³: 0.646702 ± 0.001013


In [7]:
# Caso 1: Distribución Normal con μ=0, σ=1 (estándar)
# Integral de la PDF en [-2, 2] debería ser ~0.9545 (regla 2σ)
def normalPDF(x, mu, sigma):
    coef = 1.0 / (sigma * np.sqrt(2.0 * np.pi))
    expTerm = np.exp(-0.5 * ((x[0] - mu) / sigma)**2)
    return coef * expTerm

def funcNormal(x):
    return normalPDF(x, mu=0.0, sigma=1.0)

res1, err1 = integMonteCarlo(funcNormal, dim=1, limInf=np.array([-2.0]), 
                              limSup=np.array([2.0]), numMuestras=200000, semilla=42)
print(f"Caso 1 - Normal(0,1) en [-2,2]: {res1:.6f} ± {err1:.6f} (teórico: 0.9545)")


# Caso 2: Distribución Gamma con k=2.5, θ=1.5
# PDF: f(x) = x^(k-1) * exp(-x/θ) / (θ^k * Γ(k))
def gammaPDF(x, k, theta):
    if x[0] <= 0:
        return 0.0
    num = x[0]**(k - 1.0) * np.exp(-x[0] / theta)
    den = (theta**k) * gamma(k)
    return num / den

def funcGamma(x):
    return gammaPDF(x, k=2.5, theta=1.5)

res2, err2 = integMonteCarlo(funcGamma, dim=1, limInf=np.array([0.0]), 
                              limSup=np.array([10.0]), numMuestras=200000, semilla=42)
print(f"Caso 2 - Gamma(k=2.5,θ=1.5) en [0,10]: {res2:.6f} ± {err2:.6f} (teórico: ~0.95)")


# Caso 3: Distribución Beta con α=2, β=5
# PDF: f(x) = x^(α-1) * (1-x)^(β-1) / B(α,β) para x∈[0,1]
def betaPDF(x, alpha, betaP):
    if x[0] <= 0 or x[0] >= 1:
        return 0.0
    num = x[0]**(alpha - 1.0) * (1.0 - x[0])**(betaP - 1.0)
    den = beta(alpha, betaP)
    return num / den

def funcBeta(x):
    return betaPDF(x, alpha=2.0, betaP=5.0)

res3, err3 = integMonteCarlo(funcBeta, dim=1, limInf=np.array([0.0]), 
                              limSup=np.array([1.0]), numMuestras=200000, semilla=42)
print(f"Caso 3 - Beta(α=2,β=5) en [0,1]: {res3:.6f} ± {err3:.6f} (teórico: 1.0)")


# Caso 4: Distribución t-Student con ν=3 grados de libertad
# PDF: f(x) = Γ((ν+1)/2) / [√(νπ) Γ(ν/2)] * (1 + x²/ν)^(-(ν+1)/2)
def studentTPDF(x, nu):
    coef = gamma((nu + 1.0) / 2.0) / (np.sqrt(nu * np.pi) * gamma(nu / 2.0))
    term = (1.0 + x[0]**2 / nu)**(-(nu + 1.0) / 2.0)
    return coef * term

def funcStudentT(x):
    return studentTPDF(x, nu=3.0)

res4, err4 = integMonteCarlo(funcStudentT, dim=1, limInf=np.array([-3.0]), 
                              limSup=np.array([3.0]), numMuestras=200000, semilla=42)
print(f"Caso 4 - t-Student(ν=3) en [-3,3]: {res4:.6f} ± {err4:.6f} (teórico: ~0.93)")


# Caso 5: Distribución Weibull con k=1.5 (forma), λ=2.0 (escala)
# PDF: f(x) = (k/λ) * (x/λ)^(k-1) * exp(-(x/λ)^k) para x≥0
def weibullPDF(x, k, lam):
    if x[0] < 0:
        return 0.0
    ratio = x[0] / lam
    return (k / lam) * (ratio**(k - 1.0)) * np.exp(-ratio**k)

def funcWeibull(x):
    return weibullPDF(x, k=1.5, lam=2.0)

res5, err5 = integMonteCarlo(funcWeibull, dim=1, limInf=np.array([0.0]), 
                              limSup=np.array([8.0]), numMuestras=200000, semilla=42)
print(f"Caso 5 - Weibull(k=1.5,λ=2) en [0,8]: {res5:.6f} ± {err5:.6f} (teórico: ~0.98)")


# Caso 6: Integral doble de producto de dos normales independientes
# f(x,y) = φ(x;μ₁,σ₁) * φ(y;μ₂,σ₂) sobre región rectangular
def funcNormal2D(x):
    mu1, sigma1 = 1.0, 0.8
    mu2, sigma2 = -0.5, 1.2
    pdf1 = normalPDF(np.array([x[0]]), mu1, sigma1)
    pdf2 = normalPDF(np.array([x[1]]), mu2, sigma2)
    return pdf1 * pdf2

res6, err6 = integMonteCarlo(funcNormal2D, dim=2, 
                              limInf=np.array([-1.0, -2.0]), 
                              limSup=np.array([3.0, 1.0]), 
                              numMuestras=300000, semilla=42)
print(f"Caso 6 - Producto Normal 2D en rectángulo: {res6:.6f} ± {err6:.6f}")


# Caso 7: Distribución Cauchy con x₀=0, γ=1 (colas pesadas)
# PDF: f(x) = 1 / [π γ (1 + ((x-x₀)/γ)²)]
def cauchyPDF(x, x0, gammaP):
    return 1.0 / (np.pi * gammaP * (1.0 + ((x[0] - x0) / gammaP)**2))

def funcCauchy(x):
    return cauchyPDF(x, x0=0.0, gammaP=1.0)

res7, err7 = integMonteCarlo(funcCauchy, dim=1, limInf=np.array([-5.0]), 
                              limSup=np.array([5.0]), numMuestras=200000, semilla=42)
print(f"Caso 7 - Cauchy(x₀=0,γ=1) en [-5,5]: {res7:.6f} ± {err7:.6f} (teórico: ~0.80)")

Caso 1 - Normal(0,1) en [-2,2]: 0.954592 ± 0.001030 (teórico: 0.9545)
Caso 2 - Gamma(k=2.5,θ=1.5) en [0,10]: 0.980321 ± 0.001508 (teórico: ~0.95)
Caso 3 - Beta(α=2,β=5) en [0,1]: 1.000793 ± 0.002024 (teórico: 1.0)
Caso 4 - t-Student(ν=3) en [-3,3]: 0.942651 ± 0.001561 (teórico: ~0.93)
Caso 5 - Weibull(k=1.5,λ=2) en [0,8]: 1.000156 ± 0.002376 (teórico: ~0.98)
Caso 6 - Producto Normal 2D en rectángulo: 0.779438 ± 0.001019
Caso 7 - Cauchy(x₀=0,γ=1) en [-5,5]: 0.875052 ± 0.002029 (teórico: ~0.80)
